# Train a PINN on a synthetic reef (CPU-executable)

This notebook is the Jupyter twin of [`train_on_new_reef.py`](train_on_new_reef.py). It generates a small synthetic dataset from a known thermal diffusivity and light attenuation, trains a PINN on three logger depths plus surface SST, and compares the recovered physical parameters against ground truth.

Total runtime on CPU is roughly two to four minutes. No GPU required.

**Prerequisites:** `pip install -e .[cpu]` from the repository root.

In [ ]:
import os
os.environ.setdefault('JAX_PLATFORMS', 'cpu')
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')

import numpy as np
import matplotlib.pyplot as plt

from pinn_reef_thermal import (
    ReefPINN, ReefThermalPhysics, generate_stratified_collocation_points,
)

TRUE_KAPPA = 3.5e-4
TRUE_KD = 0.12
LOGGER_DEPTHS = [2.0, 8.0, 15.0]
Z_MAX = 20.0
N_DAYS = 180

## Step 1. Generate a synthetic reef

We solve the 1D heat equation with known `kappa` and `Kd`, then sample logger observations at three depths plus daily surface SST.

In [ ]:
physics_truth = ReefThermalPhysics(
    kappa=TRUE_KAPPA, Kd=TRUE_KD,
    T_mean=28.2, T_amp=1.0,
    z_max=Z_MAX, t_days=N_DAYS,
)
z_fd, t_fd, T_fd = physics_truth.solve_fd(nz=128, dt=3600.0)

rng = np.random.default_rng(0)
t_bc = (np.arange(0, N_DAYS, 1.0) + 0.17) * 86400.0
T_bc = np.interp(t_bc, t_fd, T_fd[0]) + rng.normal(0, 0.05, size=t_bc.shape)

z_data, t_data, T_data = [], [], []
for d in LOGGER_DEPTHS:
    iz = int(np.argmin(np.abs(z_fd - d)))
    T_depth = T_fd[iz] + rng.normal(0, 0.02, size=t_fd.shape)
    z_data.append(np.full_like(t_fd, d)); t_data.append(t_fd); T_data.append(T_depth)
z_data = np.concatenate(z_data).astype(np.float32)
t_data = np.concatenate(t_data).astype(np.float32)
T_data = np.concatenate(T_data).astype(np.float32)
print(f'{len(z_data):,} logger obs, {len(t_bc):,} SST obs')

## Step 2. Configure and train the PINN

We use a smaller network (`hidden_dim=64`, `n_hidden=3`) and fewer epochs (`2000`) than the paper's full configuration to keep the notebook CPU-friendly.

In [ ]:
physics = ReefThermalPhysics(
    kappa=5e-4, Kd=0.25,
    T_mean=float(T_data.mean()), T_amp=1.0,
    z_max=Z_MAX, t_days=N_DAYS,
)
z_pde, t_pde = generate_stratified_collocation_points(
    Z_MAX, N_DAYS * 86400.0, n_points=5000,
    logger_depths=LOGGER_DEPTHS, seed=0,
)
pinn = ReefPINN(
    physics=physics, learn_kappa=True, learn_Kd=True,
    use_hard_bc=True, n_fourier=4, seed=0,
    init_kappa=1e-3, init_Kd=0.2,
    pde_chunk_size=0, pde_depth_scale=20.0,
    w_bc_bottom=0.05, kappa_mode='constant',
    t_sst=t_bc.astype(np.float32), T_sst=T_bc.astype(np.float32),
    use_modified_mlp=True, hidden_dim=64, n_hidden=3,
    T_scale=2.0, activation='tanh', grad_clip=1.0,
)
pinn.train(
    z_pde=z_pde, t_pde=t_pde,
    z_data=z_data, t_data=t_data, T_data=T_data,
    z_bc=np.zeros_like(t_bc, dtype=np.float32),
    t_bc=t_bc.astype(np.float32), T_bc=T_bc.astype(np.float32),
    n_epochs=2000, lr=1e-3, print_every=200,
    w_pde=1.0, w_data=10.0, w_bc=0.0,
)

## Step 3. Inspect recovered parameters

In [ ]:
est = pinn.get_estimated_params()
print(f"kappa:  true = {TRUE_KAPPA:.3e},  recovered = {est['kappa']:.3e}")
print(f"Kd:     true = {TRUE_KD:.3f},      recovered = {est['Kd']:.3f}")

## Step 4. Compare depth profile against ground truth

In [ ]:
t_snap = t_fd[len(t_fd) // 2]
z_grid = np.linspace(0, Z_MAX, 64)
T_pinn = np.array([float(pinn.predict(z, t_snap)) for z in z_grid])
T_true = np.interp(z_grid, z_fd, T_fd[:, len(t_fd) // 2])

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(T_true, z_grid, 'k-', lw=2, label='ground truth')
ax.plot(T_pinn, z_grid, 'tab:orange', lw=2, ls='--', label='PINN')
for d in LOGGER_DEPTHS:
    ax.axhline(d, color='tab:blue', alpha=0.3)
ax.invert_yaxis()
ax.set_xlabel('Temperature (C)'); ax.set_ylabel('Depth (m)')
ax.set_title('Synthetic reef: depth profile')
ax.legend(); ax.grid(alpha=0.3); plt.show()

## Next steps

- For real reefs, see [`extend_to_new_reef.py`](extend_to_new_reef.py), which demonstrates adapting your own logger CSVs to the expected schema.
- The paper's full experiments use `hidden_dim=128`, `n_hidden=4`, `n_epochs=15000`, and real AIMS Time Series Explorer data. Those configurations require a GPU.
- Kappa is notoriously harder to identify than Kd from surface-and-depth observations; the short training run used here recovers Kd within approximately 10 percent but only constrains kappa to within a factor of 2 to 3. Longer training and more logger depths both improve identifiability.